# 🏋️ Step 2 · Fine-tune YOLOv8-Face

**Training pipeline.** Fine-tune the pretrained face detector and register it in the model registry.

`pretrained weights → 🚀 train → 📈 evaluate → 🏷️ model registry`

> ⚙️ Requires a **GPU** (set GPU: 1 in the Jupyter config).

In [ ]:
from fedn.utils.helpers.helpers import get_helper
from ultralytics import YOLO
import torch
import collections
import numpy as np
import hopsworks
import os
from PIL import Image
import train

HELPER_MODULE = "numpyhelper"
helper = get_helper(HELPER_MODULE)

### 📦 Load data & configure training
Unpack the training data locally, load pretrained weights, and set the training params.

In [ ]:
train.copy_to_local_dir_training_data()
model = train.load_parameters("weights/face_finder_best.npz")
params = {
    'data': os.path.abspath('data/widerface.yaml'),
    'epochs': 1,
    'batch': 32,
    'imgsz': 640,
    'device': 0,
    'resume': False,
    'workers': 0,
    'cache': "ram",
    'amp': True,
}

print(params)

### 🚀 Train

In [ ]:
model.train(**params)

### 📈 Evaluate
Predict on a sample image and plot the detected faces.

In [ ]:
mr = hopsworks.login().get_model_registry()

model_dir = "mr_model"
os.makedirs(f"{model_dir}/images", exist_ok=True)
train.save_parameters(model, f"./{model_dir}/fine-tuned-model.npz")

# Sanity-check the fine-tuned model on a sample image
img_path = "data/images/bus.jpg"
results = model.predict(img_path, imgsz=640, conf=0.75, iou=0.7, device=0, verbose=False)

img = Image.fromarray(results[0].plot()[..., ::-1])  # plot() returns BGR -> flip to RGB for PIL
base = os.path.splitext(os.path.basename(img_path))[0]
output_path = os.path.abspath(f"./{model_dir}/images/{base}-faces-detected.png")
img.save(output_path, format="PNG")

print(f"✅ Detected {len(results[0].boxes)} faces — preview saved to {output_path}")

### 🏷️ Register the model
Save the fine-tuned weights and metrics to the Hopsworks model registry.

In [ ]:
metrics = {"epochs": params["epochs"], "batch": params["batch"]}

faces_model = mr.python.create_model(
    name="facerecognition",
    metrics=metrics,
    description="Yolo-v8 face recognition model",
)
faces_model.save(model_dir)

print(f"✅ Registered model '{faces_model.name}' v{faces_model.version}")